# MedRAG DeepEval walkthrough

This notebook runs a small, real evaluation against the MedRAG application. It uses the production `RAGService`, the golden dataset, and `build_rag_metrics()` (Answer Relevance, Context Relevance, Groundedness; threshold 0.7). The first two golden cases are used by default; set `MAX_CASES = None` to run all cases.

**Prerequisites:** Qdrant must be running at the configured host/port. Set `OPENAI_API_KEY` in the repository `.env`; PDF indexing also needs `LLAMA_CLOUD_API_KEY`. Indexing and DeepEval make external service calls and may incur usage costs. Do not use sensitive patient data.

The optional indexing cell uses the same project ingestor and core index builder as production. It bypasses `RAGService.build_index()` because the current `service.py` contains a `pdb.set_trace()` breakpoint. Evaluation, like the production evaluator, calls `RAGService.query()` directly and does not exercise API-boundary guardrails. Results are written to `deepeval_notebook_latest.json`, not the production `medrag_latest.json`.

## 1. Find the project and load environment

Locate the repository root from the notebook working directory, load `.env` without displaying secret values, and add the project root to Python imports.

In [1]:
import json
import os
import sys
from datetime import UTC, datetime
from pathlib import Path

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'pyproject.toml').is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError('Could not find the repository root containing pyproject.toml.')

from dotenv import load_dotenv
load_dotenv(ROOT / '.env', override=False)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f'Repository root: {ROOT}')

Repository root: f:\AI_Project\MedRAG_live-feature-io-guardrails


## 2. Create the production service and check dependencies

Fail early with a useful message if the LLM key or Qdrant server is unavailable. The notebook does not print API keys.

In [2]:
from src.core.indexer import build_index, build_qdrant_client
from src.core.projects import get_project_definition
from src.core.service import RAGService
from src.core.settings import get_settings
from src.core.evals import load_eval_dataset
from eval.shared.metrics import build_rag_metrics

settings = get_settings()
if not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError('OPENAI_API_KEY is missing. Add it to the repository .env, then restart the kernel.')

service = RAGService.from_project_name('medrag', settings=settings)
try:
    qdrant_client = build_qdrant_client(service.settings)
    qdrant_client.get_collections()
except Exception as exc:
    raise RuntimeError(
        f'Cannot connect to Qdrant at {service.settings.qdrant_host}:{service.settings.qdrant_port}. '
        'Start Qdrant or correct QDRANT_HOST/QDRANT_PORT.'
    ) from exc

print(f'Project: {service.config.name}')
print(f'Collection: {service.config.collection_name}')
print(f'Collection ready: {service.collection_ready()}')

f:\AI_Project\MedRAG_live-feature-io-guardrails\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
f:\AI_Project\MedRAG_live-feature-io-guardrails\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'validate_default' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'validate_default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(


Project: medrag
Collection: medrag_collection_bge_small
Collection ready: True


## 3. Ensure the production collection is indexed

If the collection is already present, this cell leaves it alone. If it is missing, it runs the configured MedRAG ingestion and core index builder. Reindexing replaces the collection, so do not force a rebuild unless that is intended.

In [3]:
if not service.collection_ready():
    definition = get_project_definition(service.config.name)
    pdf_paths = list((service.config.data_dir / 'guidelines').glob('*.pdf'))
    if pdf_paths and not os.getenv('LLAMA_CLOUD_API_KEY'):
        raise RuntimeError('LLAMA_CLOUD_API_KEY is required to parse the guideline PDFs.')

    ingestor = definition.ingestor_cls(service.config)
    documents = ingestor.ingest()
    if not documents:
        raise RuntimeError('No source documents were loaded; check the PDFs and ingestion settings.')

    # Same production index builder as RAGService.build_index(), without the current pdb breakpoint.
    service.index = build_index(
        documents=documents,
        config=service.config,
        settings=service.settings,
    )
    print(f'Indexed {len(documents)} source documents.')
else:
    print('Using the existing indexed collection; no rebuild performed.')

if not service.collection_ready():
    raise RuntimeError('The MedRAG collection is still unavailable after indexing.')

Using the existing indexed collection; no rebuild performed.


## 4. Load a small dataset from the production golden file

This uses the same JSON dataset as the production evaluator. Set `MAX_CASES = None` to evaluate every case.

In [8]:
MAX_CASES = 3
dataset = load_eval_dataset(service)
if MAX_CASES is not None:
    dataset = dataset[:MAX_CASES]
if not dataset:
    raise RuntimeError('The golden dataset is empty.')

print(f'Cases selected: {len(dataset)}')
for item in dataset:
    print(f"- {item['id']}: {item['query']}")

Cases selected: 3
- med_001: What is a common first-line medication approach for type 2 diabetes?
- med_002: What kinds of information do hypertension guidelines usually provide?
- med_003: Why should retrieved medical guidance include a disclaimer?


## 5. Query the real RAG service and measure DeepEval metrics

Each question goes through `RAGService.query()`. DeepEval receives the generated answer, expected answer, and retrieved context. A metric exception is recorded as a failed metric, matching the production evaluator's behavior.

In [9]:
from deepeval.test_case import LLMTestCase

started_at = datetime.now(UTC)
case_results = []

for item in dataset:
    query_result = service.query(item['query'])
    test_case = LLMTestCase(
        input=item['query'],
        actual_output=query_result.response.answer,
        expected_output=item['expected_answer'],
        retrieval_context=query_result.retrieval_context,
    )

    metric_results = []
    case_success = True
    for metric in build_rag_metrics():
        try:
            metric.measure(test_case, _show_indicator=False)
            metric_success = bool(getattr(metric, 'success', False))
            score = getattr(metric, 'score', None)
            threshold = getattr(metric, 'threshold', None)
            metric_results.append({
                'name': getattr(metric, 'name', type(metric).__name__),
                'score': float(score) if score is not None else None,
                'threshold': float(threshold) if threshold is not None else None,
                'success': metric_success,
                'reason': getattr(metric, 'reason', None),
                'error': getattr(metric, 'error', None),
            })
            case_success = case_success and metric_success
        except Exception as exc:
            threshold = getattr(metric, 'threshold', None)
            metric_results.append({
                'name': getattr(metric, 'name', type(metric).__name__),
                'score': None,
                'threshold': float(threshold) if threshold is not None else None,
                'success': False,
                'reason': None,
                'error': str(exc),
            })
            case_success = False

    case_results.append({
        'id': item['id'],
        'query': item['query'],
        'expected_answer': item['expected_answer'],
        'actual_answer': query_result.response.answer,
        'sources': query_result.response.sources,
        'retrieval_context': query_result.retrieval_context,
        'metrics': metric_results,
        'success': case_success,
    })

completed_at = datetime.now(UTC)
passed_cases = sum(case['success'] for case in case_results)
failed_cases = len(case_results) - passed_cases
summary = {
    'project': service.config.name,
    'collection_name': service.config.collection_name,
    'started_at': started_at.isoformat(),
    'completed_at': completed_at.isoformat(),
    'duration_seconds': round((completed_at - started_at).total_seconds(), 2),
    'dataset_size': len(case_results),
    'passed_cases': passed_cases,
    'failed_cases': failed_cases,
    'success_rate': round(passed_cases / len(case_results), 4) if case_results else 0.0,
    'success': failed_cases == 0,
}

print(f"Run status: {'PASS' if summary['success'] else 'FAIL'}")
print(f"Passed: {passed_cases}/{len(case_results)} | Duration: {summary['duration_seconds']}s")

Run status: FAIL
Passed: 1/3 | Duration: 60.64s


## 6. Inspect case and metric results

Review each model answer, its retrieved sources/context, and individual metric scores and errors.

In [6]:
from IPython.display import display

display(summary)
for case in case_results:
    print(f"\n{case['id']} | {'PASS' if case['success'] else 'FAIL'} | {case['query']}")
    print('Expected:', case['expected_answer'])
    print('Actual:  ', case['actual_answer'])
    print('Sources: ', case['sources'])
    display(case['metrics'])

{'project': 'medrag',
 'collection_name': 'medrag_collection_bge_small',
 'started_at': '2026-10-05T09:02:34.466849+00:00',
 'completed_at': '2026-10-05T09:03:34.126085+00:00',
 'duration_seconds': 59.66,
 'dataset_size': 2,
 'passed_cases': 1,
 'failed_cases': 1,
 'success_rate': 0.5,
 'success': False}


med_001 | FAIL | What is a common first-line medication approach for type 2 diabetes?
Expected: Clinical guidelines commonly describe metformin as a standard first-line therapy for type 2 diabetes, depending on patient-specific factors.
Actual:   A common first-line medication approach for type 2 diabetes typically includes lifestyle modification, glycemic monitoring, and the use of metformin when there are no contraindications. Individualized escalation to other medications such as GLP-1 receptor agonists, SGLT2 inhibitors, or insulin may occur based on factors like comorbidities, kidney function, cardiovascular risk, and glycemic control.
Sources:  ['Bootstrap: bootstrap_seed', 'WHO: who_bp.pdf']


[{'name': 'Answer Relevance',
  'score': 0.4444444444444444,
  'threshold': 0.7,
  'success': False,
  'reason': 'The score is 0.44 because while the output touches on relevant aspects of type 2 diabetes treatment, it diverges into discussing other medications and factors for escalation, which are not directly related to the first-line approach. This dilutes the focus on the primary question, preventing a higher score.',
  'error': None},
 {'name': 'Context Relevance',
  'score': 0.1111111111111111,
  'threshold': 0.7,
  'success': False,
  'reason': "The score is 0.11 because the majority of the context discusses topics unrelated to the input, such as 'Type 1 diabetes', 'insulin therapy', and 'hypertension'. Only one statement mentions 'Type 2 diabetes management' with 'lifestyle modification, glycemic monitoring, and metformin', which aligns with the input about 'first-line medication approach' for type 2 diabetes.",
  'error': None},
 {'name': 'Groundedness',
  'score': 1.0,
  'thre


med_002 | PASS | What kinds of information do hypertension guidelines usually provide?
Expected: Hypertension guidelines generally cover diagnosis thresholds, blood pressure targets, medication classes, and follow-up recommendations.
Actual:   Hypertension guidelines typically provide information on blood pressure thresholds, target ranges, home blood pressure monitoring, lifestyle changes, and medication classes such as ACE inhibitors, ARBs, calcium-channel blockers, and thiazide-type diuretics. They also outline follow-up cadence based on the severity of hypertension, symptoms, and treatment response. Additionally, guidelines may include recommendations for pharmacological treatment, nonpharmacological approaches, and the involvement of healthcare workers in managing hypertension.
Sources:  ['Bootstrap: bootstrap_seed', 'WHO: who_bp.pdf']


[{'name': 'Answer Relevance',
  'score': 0.9,
  'threshold': 0.7,
  'success': True,
  'reason': "The score is 0.90 because the output effectively addresses the input by providing relevant information about hypertension guidelines. However, it includes a general statement about healthcare workers that doesn't directly pertain to the specific details of the guidelines, slightly affecting the score.",
  'error': None},
 {'name': 'Context Relevance',
  'score': 0.782608695652174,
  'threshold': 0.7,
  'success': True,
  'reason': "The score is 0.78 because while the retrieval context includes relevant information about hypertension guidelines, such as 'blood pressure thresholds, target ranges, home blood pressure monitoring, lifestyle changes, and medication classes,' it also contains several irrelevant structural elements like 'Table of Contents' and 'Conclusion' that do not directly address the types of information provided by hypertension guidelines.",
  'error': None},
 {'name': 'Grou

## 7. Save a notebook-specific result

This writes a JSON report alongside the production eval results without overwriting `medrag_latest.json`.

In [7]:
result = {'summary': summary, 'cases': case_results}
output_path = service.config.data_dir / 'evals' / 'deepeval_notebook_latest.json'
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(json.dumps(result, indent=2), encoding='utf-8')
print(f'Saved notebook evaluation to: {output_path}')

Saved notebook evaluation to: F:\AI_Project\MedRAG_live-feature-io-guardrails\src\projects\medrag\data\evals\deepeval_notebook_latest.json
